In [2]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time 

In [6]:
try:
    #indicamos la url a la que realizaremos peticiones GET
    book_scrape = "https://books.toscrape.com/"
    #donde guardaremos la respuesta que obtenemos de la pagina WEB
    response = requests.get(book_scrape)
    #creamos un objeto soup para parsearlo con el contenido html 
    soup = BeautifulSoup(response.text, "html.parser")
    #verificar estado de conexion 200 = exitosa
    if response.status_code == 200:
        print("Conexion exitosa...")   
except (Exception,KeyboardInterrupt) as e:
    print(f"Hubo un error {e}")

Conexion exitosa...


In [7]:
libro_datos = [] #creamos una lista que almacenara los datos de los libros
libros_extraidos = 0
for page_num in range(1,51):

    books_pages = f'https://books.toscrape.com/catalogue/page-{page_num}.html' #realizara las iteraciones del for dentro de la url de cada pagina del catalogo
    response = requests.get(books_pages)
    soup = BeautifulSoup(response.content, 'html.parser') # obtener el html 

    libros = soup.find_all('h3')

    #iterar sobre la lista de titulos de libros
    for libro in libros:
        try:
            libro_url = libro.find('a')['href'] #encontrar la url vinculada al libro (libro.find)
            libro_response = requests.get('https://books.toscrape.com/catalogue/' + libro_url) #accede a la url de books to scrape + a la url de el libro 
            libro_soup = BeautifulSoup(libro_response.content, "html.parser")  #obtenemos el html de la pagina de la url del libro 
            
            #busqueda de datos de los libros mediante etiquetas HTML
            titulo = libro_soup.find('h1').text #encontrar el titulo del libro mediante la etiqueta del h1
            categoria = libro_soup.find('ul', class_ ="breadcrumb").find_all('a')[2].text.strip() # accede a la etiqueta ul donde encuentra la ruta de busqueda donde se encuentra la categoria del libro 
            calificacion = libro_soup.find('p', class_ ='star-rating')['class'][1]
            precio = libro_soup.find('p', class_ ="price_color").text.strip()
            disponible = libro_soup.find('p', class_="availability").text.strip()

            libros_extraidos += 1
            libro_datos.append([titulo,categoria,calificacion,precio,disponible])
        except (Exception,KeyboardInterrupt) as e:
            print(f"Hubo un error {e}")
            continue

In [ ]:
#convertir una lista en un dataframe
df = pd.DataFrame(libro_datos, columns=["titulo","categoria","calificacion","precio","disponible"])
df 

,titulo,categoria,calificacion,precio,disponible
0,A Light in the Attic,Poetry,Three,£51.77,In stock (22 available)
1,Tipping the Velvet,Historical Fiction,One,£53.74,In stock (20 available)
2,Soumission,Fiction,One,£50.10,In stock (20 available)
3,Sharp Objects,Mystery,Four,£47.82,In stock (20 available)
4,Sapiens: A Brief History of Humankind,History,Five,£54.23,In stock (20 available)
...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,Classics,One,£55.53,In stock (1 available)
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Sequential Art,Four,£57.06,In stock (1 available)
997,A Spy's Devotion (The Regency Spies of London #1),Historical Fiction,Five,£16.97,In stock (1 available)
998,1st to Die (Women's Murder Club #1),Mystery,One,£53.98,In stock (1 available)


In [10]:
# convertir dataframe a csv
df.to_csv("libros_scrapeados.csv", index=False)

print("Guardado correctamete como: libros_scrapeados.csv")

Guardado correctamete como: libros_scrapeados.csv


In [34]:
columna_titulo= ['titulo']
#leemos el csv para poder realizar acciones sobre los campos del csv
datos = pd.read_csv("../data/libros_scrapeados.csv",index_col="titulo", usecols=columna_titulo)

datos.head()

# Defining the columns to read
# usecols = ["id", "name", "host_id", "neighbourhood", "room_type", "price", "minimum_nights"]

# # Read data with subset of columns
# airbnb_data = pd.read_csv("data/listings_austin.csv", index_col="id", usecols=usecols)

# # Preview first 5 rows
# airbnb_data.head()

""
titulo
A Light in the Attic
Tipping the Velvet
Soumission
Sharp Objects
Sapiens: A Brief History of Humankind
